# Cox Proportional Hazards

### Wstęp
&emsp;Model proporcjonalnego hazardu Cox'a jest modelem pozwalającym ocenić wpływ wielu czynników na funkcję hazardu (chwilowe natężenie ryzyka). W naszym przypadku oceniać będziemy wpływ na ryzyko zapłaty (zdarzenie).

&emsp; Przy budownie modelu uwzględnimy następujące cechy:
* `total_open_amount` *(kwota faktury)*
* `days_to_due` *(po ilu dniach płatność)*
* `invoice_age` *(wiek faktury)*
* `avg_delay_customer` *(średnie opóźnienie klienta)*
* `cust_payment_terms` *(warunki płatności)*
* `invoice_currency` *(waluta)*
* `segment` *(BE lub SME)*

### Biblioteki i przygotowanie danych
&emsp; Zacznijmy od zaimportowania niezbędnych narzędzi oraz przygotowania danych. Zostawimy jedynie kolumny odpowiadające interesującym nas cechom.

In [64]:
import pandas as pd
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

df = pd.read_csv('../data/dataset_survclean.csv')

df['document_create_date_dt'] = pd.to_datetime(df['document_create_date_dt'])
df = df.sort_values('document_create_date_dt')

features = [
    'total_open_amount',
    'days_to_due',
    'invoice_age',
    'avg_delay_customer',
    'cust_payment_terms',
    'invoice_currency',
    'segment',
    'time_days', 
    'event'
]

cox_df = df[features].copy()

cox_df = pd.get_dummies(cox_df, columns=['cust_payment_terms', 'invoice_currency', 'segment'], drop_first=True)
cox_df = cox_df.dropna()
cox_df = cox_df.astype(float)

splt_id = int(len(cox_df) * 0.8)
train_df = cox_df.iloc[:splt_id]
test_df = cox_df.iloc[splt_id:]

print(f"Zbiór treningowy: {len(train_df)} faktur\nZbiór testowy: {len(test_df)} faktur")

display(train_df.head())

Zbiór treningowy: 39070 faktur
Zbiór testowy: 9768 faktur


,total_open_amount,days_to_due,invoice_age,avg_delay_customer,time_days,event,cust_payment_terms_NA10,cust_payment_terms_NA32,cust_payment_terms_NAA8,cust_payment_terms_NAAW,...,cust_payment_terms_NAVF,cust_payment_terms_NAVQ,cust_payment_terms_NAVR,cust_payment_terms_NAWN,cust_payment_terms_NAWP,cust_payment_terms_NAWU,cust_payment_terms_NAX2,cust_payment_terms_OTHER,invoice_currency_USD,segment_SME
0,13760.55,34.0,30.0,0.0,28.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
1,28225.48,32.0,30.0,0.0,62.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
53,4445.93,16.0,15.0,0.2,11.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
52,51473.49,16.0,15.0,0.0,17.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
51,4678.12,20.0,15.0,0.0,19.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


### Model
&emsp; Pozostało dopasować model Coxa do badanego zbioru danych. Kolumną zawierającą informacje na temat upływu czasu jest `time_days`, a kolumną informującą o wystąpieniu zdarzenia jest kolumna `event`.

In [65]:
cph = CoxPHFitter(penalizer=0.1)
const_cols = [c for c in train_df.columns 
              if c not in ('time_days', 'event') and train_df[c].std() == 0]
train_df = train_df.drop(columns=const_cols)

cph.fit(train_df, duration_col='time_days', event_col='event')

<lifelines.CoxPHFitter: fitted with 39070 total observations, 38 right-censored observations>

#### Załozenie o proporcjinalności hazardów
&emsp; Model Coxa opiera się na załozeniu o proporcjonalności hazardów. Załozenie to mowi, ze stosunek hazardu dla dowolnych dwoch obserwacji musi być stały w czasie. W kontekście projektu oznacza to, ze wpływ cechy na prawdopodobieństwo zapłacenia faktury nie moze się zmieniać wraz upływem dni. Wpływ ten musi być ponadto liniowy.

&emsp; Jeśli to załozenie jest łamane model traci na wiarygodności. W celu jego weryfikacji wykorzystamy test reszt Schoenfelda.

In [66]:
results = cph.check_assumptions(train_df, p_value_threshold=0.05, show_plots=False)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'total_open_amount' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'total_open_amount' might be incorrect. That is,
there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'total_open_amount' using pd.cut, and then specify it in
`strata=['total_open_amount', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'days_to_due' failed the non-proportional test: p-value is <5e-05.

   Advice 1: the functional form of the variable 'days_to_due' might be incorrect. That is, there
may be non-linear terms missing. The proportional hazard test used is very sensitive to incorrect
functional forms. See documentation in lin

&emsp;Z tabeli odczytujemy, że dla wielu zmiennych (m.in. `avg_delay_customer`, `days_to_due`, `invoice_age`, `segment_SME`, `total_open_amount`), wartość p-value jest niższa niż próg 0.05. Oznacza to, że hipoteza zerowa testu Schoenfelda jest fałszywa. Co za tym idzie, wpływ cech na "ryzyko" opłacenia faktury nie jest stały w czasie.

&emsp;Mamy do czynienia ze złamaniem podstawowego założenia modelu Coxa, co pozwala wyciągnąć wniosek, że modele nieliniowe jak Random Survival Forest poradzą sobie z predykcją zauważalnie lepiej.

#### Ewaluacja modelu
&emsp; Bazując na budowie algorytmu do wyliczenia funkcji częściowej wiarygodności (partial likelihood), w której wzorze wspolczynnik dla kazdej cechy jest stały w czasie, otrzymane w tabeli wyniki mozemy interpretować jako uśredniowy wpływ poszczególnych cech na "ryzyko"" zapłaty.

&emsp; Wywołanie `cph.print_summary` dostarczy nam informacji o m.in:
* Hazard Ratio (`exp(coef)`): Dla zmiennych kategorycznych stosunek hazardu między podmiotem posiadającym i nieposiadającym danej cechy. Dla zmiennych ciągłych zmiana intensywności spłaty przy wzroście cechy o jedną jednostkę. Interpretacja:
    * HR > 1: zwiększone ryzyko
    * HR = 1: brak wpływu
    * HR < 1: zmniejszone ryzyko
* C-index (`Concordance`): Odsetek par, które model poprawnie uporządkował pod względem czasu do wystąpienia zdarzenia.

In [67]:
cph.print_summary()
test_cid = cph.score(test_df, scoring_method="concordance_index")

print(f"test C-index: {test_cid: .4f}")

<lifelines.CoxPHFitter: fitted with 39070 total observations, 38 right-censored observations>
             duration col = 'time_days'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 39070
number of events observed = 39032
   partial log-likelihood = -361779.38
         time fit was run = 2026-09-21 16:57:06 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
total_open_amount         0.00      1.00      0.00            0.00            0.00                1.00                1.00
days_to_due              -0.04      0.96      0.00           -0.04           -0.04                0.96                0.97
invoice_age              -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
avg_delay_customer       -0.03      0.97      0.00           -0.04           -0.03                0.96                0.97
cust_payment_terms_NA10  -0.19      0.83      0.05           -0.29           -0.08                0.74                0.92
cust_payment_terms_NA32  -0.75      0.47      0.04           -0.84           -0.66                0.43                0.52
cust_payment_terms_NAA8  -0.28      0.76      0.02           -0.32           -0.24                0.73                0.79
cust_payment_terms_NAAW   0.16      1.18      0.08            0.01            0.32                1.01                1.38
cust_payment_terms_NAAX  -0.08      0.93      0.04           -0.15           -0.01                0.86                0.99
cust_payment_terms_NAC6  -0.13      0.88      0.03           -0.19           -0.07                0.82                0.93
cust_payment_terms_NAD1  -0.69      0.50      0.04           -0.76           -0.61                0.47                0.54
cust_payment_terms_NAD5  -0.87      0.42      0.06           -0.99           -0.75                0.37                0.47
cust_payment_terms_NAG2  -0.85      0.43      0.04           -0.93           -0.76                0.39                0.47
cust_payment_terms_NAGD  -0.23      0.80      0.06           -0.35           -0.10                0.71                0.90
cust_payment_terms_NAH4   0.36      1.43      0.02            0.32            0.40                1.38                1.49
cust_payment_terms_NAM1   3.81     44.94      0.09            3.63            3.98               37.90               53.28
cust_payment_terms_NAM2   3.85     46.79      0.06            3.72            3.97               41.30               53.01
cust_payment_terms_NAM4   2.13      8.44      0.04            2.06            2.21                7.84                9.09
cust_payment_terms_NAU5  -0.13      0.88      0.04           -0.20           -0.05                0.81                0.96
cust_payment_terms_NAVE  -0.80      0.45      0.07           -0.94           -0.67                0.39                0.51
cust_payment_terms_NAVF  -0.59      0.55      0.08           -0.74           -0.45                0.48                0.64
cust_payment_terms_NAVQ  -0.46      0.63      0.11           -0.67           -0.25                0.51                0.78
cust_payment_terms_NAVR   0.23      1.26      0.13           -0.02            0.48                0.98                1.61
cust_payment_terms_NAWN  -0.26      0.77      0.11           -0.48           -0.03                0.62                0.97
cust_payment_terms_NAWP  -0.78      0.46      0.12           -1.01           -0.55                0.36                0.58
cust_payment_terms_NAWU  -0.69      0.50      0.09           -0.87           -0.51                0.42                0.60
cust_payment_terms_NAX2  -1.04      0.35      0.06           -1.16           -0.93                0.31                0.39

test C-index:  0.4404


In [68]:
print(f"odsetek zdarzeń — trening: {train_df['event'].mean():.3f}, test: {test_df['event'].mean():.3f}")
print(f"C-index — trening: {cph.score(train_df, scoring_method='concordance_index'):.4f}, "
      f"test: {test_cid:.4f}")

odsetek zdarzeń — trening: 0.999, test: 0.013
C-index — trening: 0.7720, test: 0.4404


### Sprawdźmy inny podział

In [71]:
splt_id = int(len(cox_df) * 0.5)
train_df = cox_df.iloc[:splt_id]
test_df = cox_df.iloc[splt_id:]

print(f"Zbiór treningowy: {len(train_df)} faktur\nZbiór testowy: {len(test_df)} faktur")

Zbiór treningowy: 24419 faktur
Zbiór testowy: 24419 faktur


In [74]:
const_cols = [c for c in train_df.columns 
              if c not in ('time_days', 'event') and train_df[c].std() == 0]
train_df = train_df.drop(columns=const_cols)
cph.fit(train_df, duration_col='time_days', event_col='event')

cph.print_summary()
test_cid = cph.score(test_df, scoring_method="concordance_index")

print(f"test C-index: {test_cid: .4f}")

<lifelines.CoxPHFitter: fitted with 24419 total observations, 0 right-censored observations>
             duration col = 'time_days'
                event col = 'event'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 24419
number of events observed = 24419
   partial log-likelihood = -214911.02
         time fit was run = 2026-09-21 17:01:56 UTC

---
                          coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                 
total_open_amount         0.00      1.00      0.00            0.00            0.00                1.00                1.00
days_to_due              -0.03      0.97      0.00           -0.04           -0.03                0.97                0.97
invoice_age              -0.00      1.00      0.00           -0.00            0.00                1.00                1.00
avg_delay_customer       -0.03      0.97      0.00           -0.03           -0.03                0.97                0.97
cust_payment_terms_NA10  -0.21      0.81      0.07           -0.34           -0.07                0.71                0.93
cust_payment_terms_NA32  -0.75      0.47      0.06           -0.86           -0.64                0.43                0.53
cust_payment_terms_NAA8  -0.25      0.78      0.02           -0.29           -0.20                0.74                0.82
cust_payment_terms_NAAW   0.11      1.12      0.10           -0.08            0.31                0.92                1.36
cust_payment_terms_NAAX  -0.05      0.95      0.04           -0.14            0.03                0.87                1.03
cust_payment_terms_NAC6  -0.07      0.93      0.04           -0.15            0.01                0.86                1.01
cust_payment_terms_NAD1  -0.64      0.53      0.05           -0.73           -0.55                0.48                0.58
cust_payment_terms_NAD5  -0.88      0.42      0.07           -1.02           -0.73                0.36                0.48
cust_payment_terms_NAG2  -0.86      0.42      0.05           -0.96           -0.75                0.38                0.47
cust_payment_terms_NAGD  -0.30      0.74      0.07           -0.44           -0.16                0.64                0.85
cust_payment_terms_NAH4   0.36      1.44      0.03            0.31            0.41                1.37                1.51
cust_payment_terms_NAM1   3.88     48.53      0.11            3.66            4.10               38.93               60.50
cust_payment_terms_NAM2   3.88     48.62      0.08            3.72            4.05               41.35               57.17
cust_payment_terms_NAM4   2.12      8.32      0.05            2.02            2.21                7.56                9.15
cust_payment_terms_NAU5  -0.13      0.88      0.05           -0.22           -0.03                0.80                0.97
cust_payment_terms_NAVE  -0.83      0.44      0.08           -1.00           -0.67                0.37                0.51
cust_payment_terms_NAVF  -0.65      0.52      0.09           -0.83           -0.48                0.44                0.62
cust_payment_terms_NAVQ  -0.49      0.61      0.13           -0.74           -0.23                0.48                0.80
cust_payment_terms_NAVR   0.10      1.11      0.13           -0.16            0.37                0.85                1.44
cust_payment_terms_NAWN  -0.35      0.71      0.12           -0.59           -0.11                0.56                0.90
cust_payment_terms_NAWP  -0.83      0.44      0.17           -1.16           -0.49                0.31                0.61
cust_payment_terms_NAX2  -1.08      0.34      0.07           -1.21           -0.94                0.30                0.39
cust_payment_terms_OTHER -0.54      0.59      0.06           -0.65           -0.42                0.52                0.66


test C-index:  0.6133
